# The World’s Mood Map: Can News Tone Predict Protests, Conflict, and Diplomacy?

**Course:** Python Data Analytics  
**Dataset:** GDELT 2.0 Event Database  
**Core idea:** Use global news event data to explore whether media tone, event type, actor countries, and attention metrics can reveal patterns of protest, conflict, and diplomacy.

## Research Questions

1. Do countries with more negative news tone experience more conflict-related events?
2. Which event types attract the most international media attention?
3. Are diplomatic events reported with a different tone than protest or conflict events?
4. How does global event intensity change across days, weeks, and hours?
5. Can we build a simple **Global Tension Index** from news tone, event severity, and media attention?

## Data Source

GDELT 2.0 master file list:  
`http://data.gdeltproject.org/gdeltv2/masterfilelist.txt`

GDELT 2.0 event files are ZIP files ending in:  
`.export.CSV.zip`

> Important: GDELT is very large. This notebook downloads a sample of recent files, enough for a strong academic project without downloading the whole database.

In [ ]:
# ============================================================
# Cell 1: Install/import libraries
# ============================================================

import os
import re
import zipfile
import warnings
from io import BytesIO
from pathlib import Path
from datetime import datetime, timedelta

import requests
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 50)

print("Libraries loaded successfully.")

In [ ]:
# ============================================================
# Cell 2: Project configuration
# ============================================================

# Change these values depending on your computer/internet speed.
# GDELT updates every 15 minutes, so many files are available per day.

MASTER_FILE_URL = "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt"

# Number of recent GDELT export files to download.
# 8 to 20 files is usually enough for thousands of rows.
MAX_FILES = 12

# Optional: filter to specific countries.
# Use FIPS-style country codes from ActionGeo_CountryCode.
# Examples:
# ["AL"] Albania may have fewer rows depending on the time window.
# ["US", "GB", "FR", "DE", "RU", "CN"] gives richer comparison.
TARGET_COUNTRIES = None

# Minimum rows expected for the project.
MIN_ROWS_REQUIRED = 1000

# Folder to store downloaded files
DATA_DIR = Path("gdelt_data")
DATA_DIR.mkdir(exist_ok=True)

print("Configuration complete.")
print(f"Data folder: {DATA_DIR.resolve()}")

## GDELT 2.0 Event Column Names

Raw GDELT `.export.CSV.zip` files do **not always include a header row**, so we define the official event schema manually.

The most important variables for this project are:

- `AvgTone`: average tone of news coverage
- `GoldsteinScale`: event cooperation/conflict score
- `QuadClass`: broad class of event
- `NumMentions`: how often the event is mentioned
- `NumSources`: number of sources covering the event
- `ActionGeo_CountryCode`: country where the event action happened
- `EventRootCode`: high-level event type
- `SQLDATE`: event date

In [ ]:
# ============================================================
# Cell 3: Define GDELT 2.0 event column names
# ============================================================

GDELT_COLUMNS = [
    "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate",
    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
    "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
    "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode",
    "QuadClass", "GoldsteinScale", "NumMentions", "NumSources",
    "NumArticles", "AvgTone",
    "Actor1Geo_Type", "Actor1Geo_Fullname", "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code", "Actor1Geo_ADM2Code", "Actor1Geo_Lat",
    "Actor1Geo_Long", "Actor1Geo_FeatureID",
    "Actor2Geo_Type", "Actor2Geo_Fullname", "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code", "Actor2Geo_ADM2Code", "Actor2Geo_Lat",
    "Actor2Geo_Long", "Actor2Geo_FeatureID",
    "ActionGeo_Type", "ActionGeo_Fullname", "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code", "ActionGeo_ADM2Code", "ActionGeo_Lat",
    "ActionGeo_Long", "ActionGeo_FeatureID",
    "DATEADDED", "SOURCEURL"
]

# We only load the columns needed for the project to save memory.
USE_COLUMNS = [
    "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year",
    "Actor1Name", "Actor1CountryCode",
    "Actor2Name", "Actor2CountryCode",
    "EventCode", "EventBaseCode", "EventRootCode", "QuadClass",
    "GoldsteinScale", "NumMentions", "NumSources", "NumArticles", "AvgTone",
    "ActionGeo_CountryCode", "ActionGeo_Fullname", "ActionGeo_Lat", "ActionGeo_Long",
    "DATEADDED", "SOURCEURL"
]

print(f"Total schema columns: {len(GDELT_COLUMNS)}")
print(f"Columns used in this notebook: {len(USE_COLUMNS)}")

In [ ]:
# ============================================================
# Cell 4: Download master file list and select recent export files
# ============================================================

def fetch_master_file_list(master_url: str) -> pd.DataFrame:
    """Download GDELT masterfilelist.txt and return only export CSV zip files."""
    response = requests.get(master_url, timeout=60)
    response.raise_for_status()

    rows = []
    for line in response.text.splitlines():
        parts = line.strip().split()
        if len(parts) >= 3:
            size, md5, url = parts[0], parts[1], parts[2]
            if url.endswith(".export.CSV.zip"):
                filename = url.split("/")[-1]
                timestamp_match = re.search(r"(\d{14})\.export\.CSV\.zip", filename)
                if timestamp_match:
                    timestamp = pd.to_datetime(timestamp_match.group(1), format="%Y%m%d%H%M%S")
                    rows.append({
                        "size_bytes": int(size),
                        "md5": md5,
                        "url": url,
                        "filename": filename,
                        "timestamp": timestamp
                    })

    df_files = pd.DataFrame(rows).sort_values("timestamp")
    return df_files

files_df = fetch_master_file_list(MASTER_FILE_URL)

print("Available GDELT export files:", len(files_df))
print("Date range:", files_df["timestamp"].min(), "to", files_df["timestamp"].max())
files_df.tail()

In [ ]:
# ============================================================
# Cell 5: Choose recent files to download
# ============================================================

selected_files = files_df.tail(MAX_FILES).copy()

print(f"Selected {len(selected_files)} recent export files.")
selected_files[["timestamp", "size_bytes", "url"]].head()

In [ ]:
# ============================================================
# Cell 6: Download and read selected GDELT ZIP files
# ============================================================

def download_and_read_gdelt_file(url: str, data_dir: Path) -> pd.DataFrame:
    """Download one GDELT export ZIP file and read it into a DataFrame."""
    filename = url.split("/")[-1]
    local_zip_path = data_dir / filename

    # Download only if not already present
    if not local_zip_path.exists():
        print(f"Downloading {filename}...")
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        local_zip_path.write_bytes(response.content)
    else:
        print(f"Using cached file: {filename}")

    # Read zipped CSV
    with zipfile.ZipFile(local_zip_path, "r") as z:
        csv_name = z.namelist()[0]
        with z.open(csv_name) as f:
            df = pd.read_csv(
                f,
                sep="\t",
                header=None,
                names=GDELT_COLUMNS,
                usecols=USE_COLUMNS,
                dtype={
                    "GLOBALEVENTID": "Int64",
                    "SQLDATE": "Int64",
                    "Year": "Int64",
                    "QuadClass": "Int64",
                    "EventRootCode": "string",
                    "EventCode": "string",
                    "EventBaseCode": "string",
                    "Actor1CountryCode": "string",
                    "Actor2CountryCode": "string",
                    "ActionGeo_CountryCode": "string",
                    "Actor1Name": "string",
                    "Actor2Name": "string",
                    "ActionGeo_Fullname": "string",
                    "SOURCEURL": "string"
                },
                encoding="latin1",
                low_memory=False
            )
    return df

frames = []

for_url_count = 0
for url in selected_files["url"]:
    try:
        frames.append(download_and_read_gdelt_file(url, DATA_DIR))
        for_url_count += 1
    except Exception as e:
        print(f"Failed to process {url}: {e}")

raw_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print(f"Files successfully loaded: {for_url_count}")
print(f"Raw rows loaded: {len(raw_df):,}")
raw_df.head()

In [ ]:
# ============================================================
# Cell 7: Basic cleaning and feature engineering
# ============================================================

df = raw_df.copy()

# Convert dates
df["event_date"] = pd.to_datetime(df["SQLDATE"].astype(str), format="%Y%m%d", errors="coerce")
df["dateadded"] = pd.to_datetime(df["DATEADDED"].astype(str), format="%Y%m%d%H%M%S", errors="coerce")

# Numeric conversion
numeric_cols = [
    "GoldsteinScale", "NumMentions", "NumSources", "NumArticles",
    "AvgTone", "ActionGeo_Lat", "ActionGeo_Long"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows with missing essential analysis variables
essential_cols = ["event_date", "QuadClass", "GoldsteinScale", "NumMentions", "NumSources", "AvgTone"]
df = df.dropna(subset=essential_cols)

# Optional country filtering
if TARGET_COUNTRIES is not None:
    df = df[df["ActionGeo_CountryCode"].isin(TARGET_COUNTRIES)]

# Add readable QuadClass labels
quad_labels = {
    1: "Verbal Cooperation",
    2: "Material Cooperation",
    3: "Verbal Conflict",
    4: "Material Conflict"
}

df["quad_label"] = df["QuadClass"].map(quad_labels)

# Simplified event category
def classify_event(row):
    root = str(row["EventRootCode"]).zfill(2)
    quad = row["QuadClass"]

    if quad in [3, 4]:
        if root in ["14"]:
            return "Protest"
        elif root in ["18", "19", "20"]:
            return "Conflict / Violence"
        elif root in ["17"]:
            return "Coercion"
        else:
            return "Other Conflict"
    elif quad in [1, 2]:
        if root in ["03", "04"]:
            return "Diplomacy / Consultation"
        elif root in ["05"]:
            return "Engagement / Cooperation"
        elif root in ["06", "07"]:
            return "Material Cooperation"
        else:
            return "Other Cooperation"
    else:
        return "Other"

df["event_category"] = df.apply(classify_event, axis=1)

# Attention score: combines mentions, sources, and articles.
# log1p prevents very large media spikes from dominating everything.
df["attention_score"] = (
    np.log1p(df["NumMentions"]) +
    np.log1p(df["NumSources"]) +
    np.log1p(df["NumArticles"])
)

# Negative tone strength
df["negative_tone"] = np.where(df["AvgTone"] < 0, abs(df["AvgTone"]), 0)

# Conflict flag
df["is_conflict"] = df["QuadClass"].isin([3, 4]).astype(int)
df["is_material_conflict"] = (df["QuadClass"] == 4).astype(int)
df["is_diplomacy"] = df["event_category"].isin(["Diplomacy / Consultation", "Engagement / Cooperation"]).astype(int)
df["is_protest"] = (df["event_category"] == "Protest").astype(int)

# Time features
df["hour_added"] = df["dateadded"].dt.hour
df["weekday"] = df["event_date"].dt.day_name()
df["week"] = df["event_date"].dt.to_period("W").astype(str)

print(f"Cleaned rows: {len(df):,}")

if len(df) < MIN_ROWS_REQUIRED:
    print("WARNING: The dataset has fewer rows than required. Increase MAX_FILES and rerun the download cells.")

df.head()

In [ ]:
# ============================================================
# Cell 8: Dataset quality check
# ============================================================

print("Shape:", df.shape)
print("\nMissing values in selected columns:")
print(df[USE_COLUMNS].isna().mean().sort_values(ascending=False).head(15))

print("\nEvent date range:")
print(df["event_date"].min(), "to", df["event_date"].max())

print("\nUnique countries:", df["ActionGeo_CountryCode"].nunique())
print("\nQuadClass distribution:")
print(df["quad_label"].value_counts(dropna=False))

In [ ]:
# ============================================================
# Cell 9: Descriptive statistics
# ============================================================

summary_cols = ["AvgTone", "GoldsteinScale", "NumMentions", "NumSources", "NumArticles", "attention_score"]
df[summary_cols].describe().T

# Analysis Question 1

## Do countries with more negative news tone experience more conflict-related events?

We aggregate by country and compare:

- average tone
- conflict event share
- material conflict share
- total number of events
- average Goldstein Scale
- average media attention

In [ ]:
# ============================================================
# Cell 10: Country-level mood/conflict table
# ============================================================

country_stats = (
    df.dropna(subset=["ActionGeo_CountryCode"])
      .groupby("ActionGeo_CountryCode")
      .agg(
          events=("GLOBALEVENTID", "count"),
          avg_tone=("AvgTone", "mean"),
          median_tone=("AvgTone", "median"),
          conflict_share=("is_conflict", "mean"),
          material_conflict_share=("is_material_conflict", "mean"),
          protest_share=("is_protest", "mean"),
          diplomacy_share=("is_diplomacy", "mean"),
          avg_goldstein=("GoldsteinScale", "mean"),
          avg_attention=("attention_score", "mean"),
          total_mentions=("NumMentions", "sum")
      )
      .reset_index()
)

# Keep countries with enough observations for reliable comparison
country_stats = country_stats[country_stats["events"] >= 20].copy()

country_stats.sort_values("conflict_share", ascending=False).head(10)

In [ ]:
# ============================================================
# Cell 11: Correlation between tone and conflict indicators
# ============================================================

corr_cols = ["avg_tone", "conflict_share", "material_conflict_share", "protest_share", "avg_goldstein", "avg_attention"]
country_corr = country_stats[corr_cols].corr(method="spearman")

plt.figure(figsize=(10, 7))
sns.heatmap(country_corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Country-Level Spearman Correlation: Tone, Conflict, Protest, Diplomacy")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 12: Scatter plot — average tone vs conflict share
# ============================================================

fig = px.scatter(
    country_stats,
    x="avg_tone",
    y="conflict_share",
    size="events",
    hover_name="ActionGeo_CountryCode",
    hover_data=["events", "avg_goldstein", "avg_attention", "protest_share"],
    title="Do Countries with More Negative News Tone Have More Conflict Events?",
    labels={
        "avg_tone": "Average News Tone",
        "conflict_share": "Share of Conflict Events",
        "events": "Number of Events"
    }
)

fig.show()

In [ ]:
# ============================================================
# Cell 13: Choropleth map — average tone by country
# ============================================================

fig = px.choropleth(
    country_stats,
    locations="ActionGeo_CountryCode",
    color="avg_tone",
    hover_name="ActionGeo_CountryCode",
    hover_data=["events", "conflict_share", "avg_goldstein"],
    title="World Mood Map: Average News Tone by Country",
    color_continuous_scale="RdBu"
)

fig.show()

# Analysis Question 2

## Which event types attract the most international media attention?

We rank event categories using:

- total events
- total mentions
- average number of sources
- average attention score
- average tone
- average Goldstein Scale

In [ ]:
# ============================================================
# Cell 14: Event-category attention table
# ============================================================

category_stats = (
    df.groupby("event_category")
      .agg(
          events=("GLOBALEVENTID", "count"),
          avg_tone=("AvgTone", "mean"),
          avg_goldstein=("GoldsteinScale", "mean"),
          avg_mentions=("NumMentions", "mean"),
          total_mentions=("NumMentions", "sum"),
          avg_sources=("NumSources", "mean"),
          avg_articles=("NumArticles", "mean"),
          avg_attention=("attention_score", "mean"),
          conflict_share=("is_conflict", "mean")
      )
      .reset_index()
      .sort_values("avg_attention", ascending=False)
)

category_stats

In [ ]:
# ============================================================
# Cell 15: Bubble chart — media attention by event category
# ============================================================

fig = px.scatter(
    category_stats,
    x="avg_goldstein",
    y="avg_tone",
    size="total_mentions",
    color="event_category",
    hover_name="event_category",
    hover_data=["events", "avg_attention", "conflict_share"],
    title="Which Event Categories Attract the Most Media Attention?",
    labels={
        "avg_goldstein": "Average Goldstein Scale",
        "avg_tone": "Average News Tone",
        "total_mentions": "Total Mentions"
    }
)

fig.show()

In [ ]:
# ============================================================
# Cell 16: Bar chart — top event root codes by total mentions
# ============================================================

root_stats = (
    df.groupby("EventRootCode")
      .agg(
          events=("GLOBALEVENTID", "count"),
          total_mentions=("NumMentions", "sum"),
          avg_tone=("AvgTone", "mean"),
          avg_goldstein=("GoldsteinScale", "mean")
      )
      .reset_index()
      .sort_values("total_mentions", ascending=False)
      .head(15)
)

plt.figure(figsize=(12, 6))
sns.barplot(data=root_stats, x="EventRootCode", y="total_mentions")
plt.title("Top Event Root Codes by Total Media Mentions")
plt.xlabel("Event Root Code")
plt.ylabel("Total Mentions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

root_stats

# Analysis Question 3

## Are diplomatic events reported with a different tone than protest or conflict events?

Here we compare distributions of `AvgTone` and `GoldsteinScale` across event categories.

In [ ]:
# ============================================================
# Cell 17: Distribution of news tone by event category
# ============================================================

major_categories = category_stats[category_stats["events"] >= 50]["event_category"].tolist()
df_major = df[df["event_category"].isin(major_categories)].copy()

plt.figure(figsize=(14, 7))
sns.violinplot(
    data=df_major,
    x="event_category",
    y="AvgTone",
    inner="quartile",
    cut=0
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("News Tone Distribution by Event Category")
plt.xlabel("Event Category")
plt.ylabel("Average Tone")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 18: Boxplot of Goldstein Scale by event category
# ============================================================

plt.figure(figsize=(14, 7))
sns.boxplot(
    data=df_major,
    x="event_category",
    y="GoldsteinScale"
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Goldstein Scale by Event Category")
plt.xlabel("Event Category")
plt.ylabel("Goldstein Scale")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 19: Summary comparison: diplomacy vs protest vs conflict
# ============================================================

comparison_groups = df[df["event_category"].isin([
    "Diplomacy / Consultation",
    "Engagement / Cooperation",
    "Protest",
    "Conflict / Violence",
    "Coercion"
])].copy()

comparison_summary = (
    comparison_groups.groupby("event_category")
    .agg(
        events=("GLOBALEVENTID", "count"),
        avg_tone=("AvgTone", "mean"),
        median_tone=("AvgTone", "median"),
        avg_goldstein=("GoldsteinScale", "mean"),
        avg_mentions=("NumMentions", "mean"),
        avg_sources=("NumSources", "mean")
    )
    .reset_index()
    .sort_values("avg_goldstein")
)

comparison_summary

# Analysis Question 4

## How does global event intensity change across time?

We study:

- daily volume of events
- daily average tone
- daily conflict share
- hourly activity patterns

In [ ]:
# ============================================================
# Cell 20: Daily time-series table
# ============================================================

daily = (
    df.groupby("event_date")
      .agg(
          events=("GLOBALEVENTID", "count"),
          avg_tone=("AvgTone", "mean"),
          conflict_share=("is_conflict", "mean"),
          protest_events=("is_protest", "sum"),
          diplomacy_events=("is_diplomacy", "sum"),
          total_mentions=("NumMentions", "sum"),
          avg_attention=("attention_score", "mean")
      )
      .reset_index()
      .sort_values("event_date")
)

daily["rolling_avg_tone"] = daily["avg_tone"].rolling(3, min_periods=1).mean()
daily["rolling_conflict_share"] = daily["conflict_share"].rolling(3, min_periods=1).mean()

daily.head()

In [ ]:
# ============================================================
# Cell 21: Line chart — events and tone over time
# ============================================================

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily["event_date"],
    y=daily["events"],
    mode="lines+markers",
    name="Event Volume",
    yaxis="y1"
))

fig.add_trace(go.Scatter(
    x=daily["event_date"],
    y=daily["rolling_avg_tone"],
    mode="lines+markers",
    name="Rolling Avg Tone",
    yaxis="y2"
))

fig.update_layout(
    title="Global Event Volume and News Tone Over Time",
    xaxis=dict(title="Date"),
    yaxis=dict(title="Number of Events"),
    yaxis2=dict(title="Average Tone", overlaying="y", side="right"),
    legend=dict(x=0.01, y=0.99)
)

fig.show()

In [ ]:
# ============================================================
# Cell 22: Heatmap — events by weekday and hour
# ============================================================

hour_weekday = (
    df.dropna(subset=["hour_added", "weekday"])
      .groupby(["weekday", "hour_added"])
      .size()
      .reset_index(name="events")
)

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

hour_pivot = hour_weekday.pivot(index="weekday", columns="hour_added", values="events")
hour_pivot = hour_pivot.reindex(weekday_order)

plt.figure(figsize=(15, 6))
sns.heatmap(hour_pivot, cmap="YlOrRd")
plt.title("GDELT Event Volume by Weekday and Hour Added")
plt.xlabel("Hour Added")
plt.ylabel("Weekday")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 23: Stacked area chart — event category over time
# ============================================================

category_time = (
    df.groupby(["event_date", "event_category"])
      .size()
      .reset_index(name="events")
)

category_pivot = category_time.pivot(
    index="event_date",
    columns="event_category",
    values="events"
).fillna(0)

fig = px.area(
    category_pivot.reset_index(),
    x="event_date",
    y=category_pivot.columns,
    title="Event Categories Over Time",
    labels={"value": "Number of Events", "event_date": "Date"}
)

fig.show()

# Bonus Analysis

## Global Tension Index

We create a simple index using:

- negative tone
- conflict share
- material conflict share
- media attention
- inverse Goldstein score

This is not an official index. It is a student-created analytical metric for storytelling and ranking.

In [ ]:
# ============================================================
# Cell 24: Build country-level Global Tension Index
# ============================================================

tension = country_stats.copy()

# Components
tension["negative_tone_score"] = np.maximum(-tension["avg_tone"], 0)
tension["inverse_goldstein"] = np.maximum(-tension["avg_goldstein"], 0)

index_features = [
    "negative_tone_score",
    "conflict_share",
    "material_conflict_share",
    "avg_attention",
    "inverse_goldstein"
]

# Standardize features before combining
scaler = StandardScaler()
scaled = scaler.fit_transform(tension[index_features].fillna(0))

tension["global_tension_index"] = scaled.mean(axis=1)

# Rescale to 0-100 for presentation
min_val = tension["global_tension_index"].min()
max_val = tension["global_tension_index"].max()
tension["global_tension_index_0_100"] = (
    100 * (tension["global_tension_index"] - min_val) / (max_val - min_val)
)

tension_ranked = tension.sort_values("global_tension_index_0_100", ascending=False)

tension_ranked.head(15)

In [ ]:
# ============================================================
# Cell 25: Bar chart — top countries by Global Tension Index
# ============================================================

top_tension = tension_ranked.head(15)

plt.figure(figsize=(12, 7))
sns.barplot(
    data=top_tension,
    x="global_tension_index_0_100",
    y="ActionGeo_CountryCode"
)
plt.title("Top Countries by Student-Created Global Tension Index")
plt.xlabel("Global Tension Index, 0-100")
plt.ylabel("Country Code")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 26: Choropleth — Global Tension Index
# ============================================================

fig = px.choropleth(
    tension,
    locations="ActionGeo_CountryCode",
    color="global_tension_index_0_100",
    hover_name="ActionGeo_CountryCode",
    hover_data=["events", "avg_tone", "conflict_share", "avg_goldstein", "avg_attention"],
    title="Student-Created Global Tension Index by Country",
    color_continuous_scale="Reds"
)

fig.show()

# Optional Machine Learning Extension

## Country Clustering

We cluster countries based on:

- average tone
- conflict share
- protest share
- diplomacy share
- average Goldstein Scale
- media attention

This can identify groups such as:

- high-conflict / high-attention countries
- low-conflict / diplomatic countries
- negative-tone but low-event countries

In [ ]:
# ============================================================
# Cell 27: KMeans clustering of country profiles
# ============================================================

cluster_features = [
    "avg_tone",
    "conflict_share",
    "material_conflict_share",
    "protest_share",
    "diplomacy_share",
    "avg_goldstein",
    "avg_attention"
]

cluster_df = country_stats.dropna(subset=cluster_features).copy()

# Need enough countries for clustering
if len(cluster_df) >= 6:
    X = cluster_df[cluster_features]
    X_scaled = StandardScaler().fit_transform(X)

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    cluster_df["cluster"] = kmeans.fit_predict(X_scaled)

    fig = px.scatter(
        cluster_df,
        x="avg_tone",
        y="conflict_share",
        size="events",
        color="cluster",
        hover_name="ActionGeo_CountryCode",
        hover_data=["avg_goldstein", "avg_attention", "protest_share", "diplomacy_share"],
        title="Country Clusters Based on Tone, Conflict, Diplomacy, and Attention"
    )
    fig.show()

    display(cluster_df.sort_values(["cluster", "conflict_share"], ascending=[True, False]).head(20))
else:
    print("Not enough countries for clustering. Increase MAX_FILES and rerun the notebook.")

# Final Interpretation Helper

Use this section to generate text you can adapt for your report or poster.

In [ ]:
# ============================================================
# Cell 28: Auto-generate interpretation notes
# ============================================================

notes = []

if len(country_stats) > 0:
    most_negative = country_stats.sort_values("avg_tone").iloc[0]
    most_conflict = country_stats.sort_values("conflict_share", ascending=False).iloc[0]
    highest_attention_category = category_stats.sort_values("avg_attention", ascending=False).iloc[0]

    notes.append(
        f"The country with the lowest average news tone in this sample is "
        f"{most_negative['ActionGeo_CountryCode']} with avg tone {most_negative['avg_tone']:.2f}."
    )
    notes.append(
        f"The country with the highest conflict-event share is "
        f"{most_conflict['ActionGeo_CountryCode']} with conflict share {most_conflict['conflict_share']:.2%}."
    )
    notes.append(
        f"The event category with the highest average attention score is "
        f"{highest_attention_category['event_category']}."
    )

print("POSTER / REPORT INTERPRETATION NOTES")
print("------------------------------------")
for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

# Academic Poster Structure

## Suggested Poster Sections

1. **Problem:** Can global news tone act as an early signal of protest, conflict, or diplomacy?
2. **Dataset:** GDELT 2.0 event database, sampled from recent `.export.CSV.zip` files.
3. **Methods:** Cleaning, event classification, aggregation by country/time/category, correlation, visualization, custom index.
4. **Main Findings:** Use your strongest 3 visuals from this notebook.
5. **Global Tension Index:** Present as your original contribution.
6. **Limitations:** GDELT reflects media reporting, not perfect ground truth. Some countries receive more coverage than others.
7. **Conclusion:** News tone is not just text sentiment; it can become a measurable social signal.

In [ ]:
# ============================================================
# Cell 29: Export cleaned data and summary tables
# ============================================================

OUTPUT_DIR = Path("gdelt_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df.to_csv(OUTPUT_DIR / "cleaned_gdelt_events_sample.csv", index=False)
country_stats.to_csv(OUTPUT_DIR / "country_mood_conflict_stats.csv", index=False)
category_stats.to_csv(OUTPUT_DIR / "event_category_attention_stats.csv", index=False)
daily.to_csv(OUTPUT_DIR / "daily_event_intensity_stats.csv", index=False)
tension_ranked.to_csv(OUTPUT_DIR / "global_tension_index_country_ranking.csv", index=False)

print("Export complete.")
print(f"Files saved in: {OUTPUT_DIR.resolve()}")

# Conclusion Template

You can adapt this text:

> This project used GDELT 2.0 news event data to examine whether global news tone is related to protest, conflict, diplomacy, and media attention. The results show that tone, Goldstein Scale, event category, and media attention can be combined to identify countries and event types that appear more tense or more cooperative in the news ecosystem. However, because GDELT is based on media reporting, the findings should be interpreted as patterns of reported global events rather than perfect measurements of reality.

## Strongest Visuals to Put on the Poster

1. World Mood Map: Average News Tone by Country  
2. Scatter Plot: Average Tone vs. Conflict Share  
3. Bubble Chart: Event Category by Tone, Goldstein Scale, and Mentions  
4. Violin Plot: News Tone by Event Category  
5. Heatmap: Events by Weekday and Hour  
6. Global Tension Index Map